STEP 5: Build SaaS Product Analytics Dashboard in Power BI

You already calculated:

MAU

MRR

Churn

ARPU

Funnel metrics

Cohort retention

Now we visualize them.

In [3]:
import pandas as pd
from pathlib import Path

# ============================================
# 1. LOAD SOURCE DATASETS
# ============================================

users = pd.read_csv("users.csv")
subscriptions = pd.read_csv("subscriptions.csv")
events = pd.read_csv("events.csv")


# ============================================
# 2. PREPARE USERS
# ============================================

users["signup_date"] = pd.to_datetime(users["signup_date"])

users["signup_month"] = (
    users["signup_date"]
    .dt.to_period("M")
    .astype(str)
)


# ============================================
# 3. PREPARE SUBSCRIPTIONS
# ============================================

subscriptions["start_date"] = pd.to_datetime(
    subscriptions["start_date"]
)

subscriptions["end_date"] = pd.to_datetime(
    subscriptions["end_date"],
    errors="coerce"
)

subscriptions["is_churned"] = (
    subscriptions["end_date"].notna()
)

subscriptions = subscriptions.rename(
    columns={
        "start_date": "subscription_start",
        "end_date": "subscription_end"
    }
)


# ============================================
# 4. PREPARE EVENTS
# ============================================

events["event_date"] = pd.to_datetime(
    events["event_date"]
)


# ============================================
# 5. EVENT SUMMARY PER USER
# ============================================

event_summary = (
    events
    .groupby("user_id")
    .agg(
        total_events=("event_type", "count"),

        total_session_minutes=(
            "session_duration",
            "sum"
        ),

        avg_session_duration=(
            "session_duration",
            "mean"
        ),

        first_event_date=(
            "event_date",
            "min"
        ),

        last_event_date=(
            "event_date",
            "max"
        )
    )
    .reset_index()
)


# ============================================
# 6. EVENT TYPE COUNTS
# ============================================

event_counts = (
    events
    .pivot_table(
        index="user_id",
        columns="event_type",
        values="event_date",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

event_counts = event_counts.rename(
    columns={
        "login": "login_events",
        "feature_use": "feature_use_events",
        "upgrade": "upgrade_events",
        "cancel": "cancel_events"
    }
)


# ============================================
# 7. MERGE EVENT INFORMATION
# ============================================

event_summary = event_summary.merge(
    event_counts,
    on="user_id",
    how="left"
)


# Make sure all event columns exist
for col in [
    "login_events",
    "feature_use_events",
    "upgrade_events",
    "cancel_events"
]:
    if col not in event_summary.columns:
        event_summary[col] = 0


# ============================================
# 8. BUILD USER 360
# ============================================

user_360 = (
    users
    .merge(
        subscriptions[
            [
                "user_id",
                "plan_type",
                "monthly_price",
                "subscription_start",
                "subscription_end",
                "is_churned"
            ]
        ],
        on="user_id",
        how="left"
    )
    .merge(
        event_summary,
        on="user_id",
        how="left"
    )
)


# ============================================
# 9. HANDLE MISSING VALUES
# ============================================

numeric_columns = [
    "total_events",
    "login_events",
    "feature_use_events",
    "upgrade_events",
    "cancel_events",
    "total_session_minutes"
]

user_360[numeric_columns] = (
    user_360[numeric_columns]
    .fillna(0)
)

user_360["avg_session_duration"] = (
    user_360["avg_session_duration"]
    .fillna(0)
)


# ============================================
# 10. ROUND METRICS
# ============================================

user_360["total_session_minutes"] = (
    user_360["total_session_minutes"]
    .round(2)
)

user_360["avg_session_duration"] = (
    user_360["avg_session_duration"]
    .round(2)
)


# ============================================
# 11. FINAL COLUMN ORDER
# ============================================

user_360 = user_360[
    [
        "user_id",
        "signup_date",
        "signup_month",
        "country",
        "acquisition_channel",

        "plan_type",
        "monthly_price",
        "subscription_start",
        "subscription_end",
        "is_churned",

        "total_events",
        "login_events",
        "feature_use_events",
        "upgrade_events",
        "cancel_events",

        "total_session_minutes",
        "avg_session_duration",
        "first_event_date",
        "last_event_date"
    ]
]


# ============================================
# 12. VALIDATION
# ============================================

print("USER 360 DATASET")
print("=" * 50)

print(f"Rows: {len(user_360):,}")
print(f"Columns: {len(user_360.columns)}")
print(f"Unique Users: {user_360['user_id'].nunique():,}")
print(
    f"Duplicate User IDs: "
    f"{user_360['user_id'].duplicated().sum():,}"
)

print("\nColumns:")
print(user_360.columns.tolist())

print("\nPreview:")
display(user_360.head())


# ============================================
# 13. EXPORT
# ============================================

user_360.to_csv(
    "user_360.csv",
    index=False
)

print("\n✅ user_360.csv created successfully!")

USER 360 DATASET
Rows: 2,000
Columns: 19
Unique Users: 2,000
Duplicate User IDs: 0

Columns:
['user_id', 'signup_date', 'signup_month', 'country', 'acquisition_channel', 'plan_type', 'monthly_price', 'subscription_start', 'subscription_end', 'is_churned', 'total_events', 'login_events', 'feature_use_events', 'upgrade_events', 'cancel_events', 'total_session_minutes', 'avg_session_duration', 'first_event_date', 'last_event_date']

Preview:


,user_id,signup_date,signup_month,country,acquisition_channel,plan_type,monthly_price,subscription_start,subscription_end,is_churned,total_events,login_events,feature_use_events,upgrade_events,cancel_events,total_session_minutes,avg_session_duration,first_event_date,last_event_date
0,1,2026-06-06,2026-06,Germany,Referral,Free,0,2026-06-06,NaT,False,24,3,7,9,5,326.48,13.60,2026-06-10,2026-11-14
1,2,2026-09-25,2026-09,USA,Referral,Free,0,2026-09-25,2026-12-07,True,28,9,6,8,5,428.57,15.31,2026-10-18,2027-03-20
2,3,2026-01-14,2026-01,Germany,Paid Ads,Free,0,2026-01-14,NaT,False,22,4,5,8,5,353.24,16.06,2026-01-15,2026-07-10
3,4,2026-06-01,2026-06,Germany,Referral,Free,0,2026-06-01,NaT,False,6,2,1,2,1,112.37,18.73,2026-06-07,2026-10-30
4,5,2026-03-28,2026-03,UK,Organic,Free,0,2026-03-28,NaT,False,18,7,1,4,6,337.49,18.75,2026-04-03,2026-09-11



✅ user_360.csv created successfully!
